# Fake.br: experimento One-Class SVM para desvio linguístico

Notebook independente derivado do protocolo do Isolation Forest. O detector aprende somente
com notícias **True**; labels Fake aparecem apenas na seleção assistida por validação e na
avaliação externa. `anomalyScore` não é probabilidade de falsidade e um alerta não comprova
que a notícia seja falsa.

**Recorte implementado:** matriz fechada de quatro valores de `nu`, seleção ROC-AUC → AP →
menor `nu` na validação, decisões nativa e q95 separadas, avaliação única no teste e controle
Isolation Forest no mesmo split. Visualizações completas e casos extremos ficam fora deste recorte.


## 1. Imports e configuração do ambiente

Execute esta célula em um kernel limpo. No Colab, as bibliotecas `numpy`, `pandas`, `scikit-learn` e `matplotlib` normalmente já estão disponíveis; se o ambiente solicitar, instale-as antes de executar todas as células.

In [ ]:
import hashlib
import json
import os
import platform
import re
import time
import unicodedata
from importlib.metadata import version
from itertools import combinations
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

In [ ]:
RANDOM_STATE = 42
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Python:", platform.python_version())
print({name: version(name) for name in ["numpy", "pandas", "scikit-learn", "matplotlib", "ipykernel"]})
BASE_NOTEBOOK_SHA256 = "25ff9d35c233c9bd6067ce0ebcf14bc78a077659f5c769a15b8f4486bd542d79"
EXPERIMENT_KEY = "one-class-svm-core-confirmatory"


## 3. Carregamento dos dados

Mesmo ZIP e revisão fixa do notebook original. Textos completos e metadados são
carregados para auditoria; o truncamento acontece ANTES da extração usada nos
modelos. Nenhuma notícia é excluída por comprimento.

In [ ]:
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
project_folder = Path.cwd()
data_folder = project_folder / "data"
data_folder.mkdir(exist_ok=True)
archive_path = data_folder / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"

if not archive_path.exists():
    temporary_path = archive_path.with_suffix(".download")
    with urlopen(CORPUS_URL, timeout=120) as response, temporary_path.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    with ZipFile(temporary_path) as archive:
        assert archive.testzip() is None, "ZIP corrompido; refaça o download."
    temporary_path.replace(archive_path)

print("Corpus revision:", CORPUS_REVISION)
print("Archive SHA256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())

In [ ]:
metadata_columns = [
    "autor", "link", "categoria", "data_publicacao",
    "num_tokens", "num_palavras", "num_types", "num_links", "num_maiusculas",
    "num_verbos", "num_verbos_subj_imp", "num_substantivos", "num_adjetivos",
    "num_adverbios", "num_verbos_modais", "num_pron_1_2_sing", "num_pron_1_plural",
    "num_pronomes", "pausalidade", "num_caracteres", "tam_medio_sentenca",
    "tam_medio_palavra", "pct_erros_ortograficos", "emotividade", "diversidade",
]

In [ ]:
def load_news_texts(archive, folder, label):
    records = []
    text_folder = f"/full_texts/{folder}/"
    for name in sorted(archive.namelist()):
        if text_folder not in name or not name.endswith(".txt"):
            continue
        article_id = Path(name).stem + ("t" if label == 0 else "")
        records.append({
            "id": article_id,
            "text": archive.read(name).decode("utf-8"),
            "label": label,
            "sourceClass": folder,
        })
    assert records, f"Nenhum texto encontrado em {folder}"
    return pd.DataFrame(records)


def load_news_metadata(archive, folder, label):
    records = []
    metadata_folder = f"/full_texts/{folder}-meta-information/"
    for name in sorted(archive.namelist()):
        if metadata_folder not in name or not name.endswith("-meta.txt"):
            continue
        values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
        assert len(values) == len(metadata_columns), (
            f"Esquema inesperado em {name}: {len(values)} linhas"
        )
        article_id = Path(name).name.removesuffix("-meta.txt") + ("t" if label == 0 else "")
        records.append({
            **dict(zip(metadata_columns, values)),
            "id": article_id,
            "metadataLabel": label,
        })
    assert records, f"Nenhum metadado encontrado em {folder}"
    return pd.DataFrame(records)

In [ ]:
with ZipFile(archive_path) as archive:
    texts_frame = pd.concat([
        load_news_texts(archive, "fake", 1), load_news_texts(archive, "true", 0),
    ], ignore_index=True)
    metadata_frame = pd.concat([
        load_news_metadata(archive, "fake", 1), load_news_metadata(archive, "true", 0),
    ], ignore_index=True)

assert texts_frame["id"].is_unique and metadata_frame["id"].is_unique
assert set(texts_frame["id"]) == set(metadata_frame["id"]), "Texto/metadados sem correspondência"
news_frame = texts_frame.merge(metadata_frame, on="id", validate="one_to_one", indicator=True)
assert news_frame["_merge"].eq("both").all()
assert news_frame["label"].eq(news_frame["metadataLabel"]).all()
news_frame = news_frame.drop(columns=["_merge", "metadataLabel"])
assert news_frame["text"].str.strip().ne("").all()
print(f"{len(news_frame):,} notícias carregadas; textos e metadados correspondem 1:1.")

## 4. Contrato dos labels

**0 = True; 1 = Fake.** O rótulo forma as partições e permite avaliação externa.
Não entra como feature. A origem nas pastas também é verificada.

In [ ]:
label_names = {0: "True", 1: "Fake"}
assert label_names == {0: "True", 1: "Fake"}
assert set(news_frame["label"].unique()) == {0, 1}
assert news_frame.loc[news_frame["label"].eq(0), "sourceClass"].eq("true").all()
assert news_frame.loc[news_frame["label"].eq(1), "sourceClass"].eq("fake").all()
display(news_frame.groupby(["label", "sourceClass"]).size().rename("newsCount").to_frame())

## 5. Preparação dos metadados

Preservamos todos os campos brutos em `raw_news_frame` e todas as contagens numéricas
em `news_frame`. Valores não numéricos viram NaN com contagem explícita; nenhum
ausente é preenchido antes do treino. `tem_autor` segue a lógica do original:
ausência para string vazia, `None`, `none` ou `NULL`. Não é uma medida de credibilidade.

In [ ]:
raw_news_frame = news_frame.copy(deep=True)
numeric_metadata_columns = metadata_columns[4:]
converted_metadata = news_frame[numeric_metadata_columns].apply(pd.to_numeric, errors="coerce")
conversion_missing = converted_metadata.isna().sum().rename("missingAfterNumericConversion")
display(conversion_missing.to_frame())
news_frame[numeric_metadata_columns] = converted_metadata
news_frame["tem_autor"] = (~news_frame["autor"].fillna("").astype(str).str.strip().isin(
    ["", "None", "none", "NULL"]
)).astype(int)

## 6. Extração no texto efetivamente utilizado

Normalização Unicode NFKC, remoção do BOM inicial e primeiros CHARACTER_LIMIT
caracteres, incluindo espaços/pontuação. Textos menores permanecem menores, sem
preenchimento. O corte pode dividir palavras/frases. Palavras são sequências de
letras com hífen/apóstrofo interno; tokens também incluem números e pontuação.
Tipos são palavras distintas ignorando caixa. TTR = tipos/tokens; diversidade =
tipos/palavras. Maiúsculas conta palavras totalmente maiúsculas com mais de uma
letra. Links são URLs http(s) presentes no corpo. Estas definições explícitas
não pretendem reproduzir o extrator desconhecido dos metadados históricos.

As demais contagens linguísticas são marcadas ausentes na cópia de features,
não imputadas nem selecionadas pelo modelo; originais ficam em raw_news_frame e
news_frame. Autor permanece como metadado válido da notícia inteira.

In [ ]:
def calculate_ratio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").astype(float)
    denominator = pd.to_numeric(denominator, errors="coerce").astype(float)
    safe_denominator = denominator.where(denominator.gt(0) & np.isfinite(denominator))
    return numerator.div(safe_denominator).replace([np.inf, -np.inf], np.nan)

In [ ]:
CHARACTER_LIMIT = 300
numeric_metadata_columns = metadata_columns[4:]
ANOMALY_COLUMNS = [
    "tem_autor",
    "typeTokenRatio",
    "linkDensity",
    "punctuationDensity",
    "uppercaseRatio",
    "diversidade",
]
WORD_PATTERN = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
TOKEN_PATTERN = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)


def measure_text(text, character_limit):
    normalized_text = unicodedata.normalize("NFKC", text).lstrip("\ufeff")
    if character_limit is not None:
        if not isinstance(character_limit, int) or character_limit < 1:
            raise ValueError("Limite deve ser inteiro positivo ou None.")
        normalized_text = normalized_text[:character_limit]
    words = WORD_PATTERN.findall(normalized_text)
    tokens = TOKEN_PATTERN.findall(normalized_text)
    return {
        "text": normalized_text,
        "num_palavras": len(words),
        "num_tokens": len(tokens),
        "num_types": len({word.casefold() for word in words}),
        "num_links": len(re.findall(r"https?://\S+", normalized_text, flags=re.IGNORECASE)),
        "num_maiusculas": sum(word.isupper() and len(word) > 1 for word in words),
        "num_caracteres": len(normalized_text),
    }


def add_derived_ratio_features(features_frame):
    ratio_columns = [
        ("typeTokenRatio", "num_types", "num_tokens"),
        ("diversidade", "num_types", "num_palavras"),
        ("linkDensity", "num_links", "num_palavras"),
    ]
    for ratio_column, numerator_column, denominator_column in ratio_columns:
        features_frame[ratio_column] = calculate_ratio(
            features_frame[numerator_column], features_frame[denominator_column]
        )
    features_frame["punctuationDensity"] = calculate_ratio(
        features_frame["num_tokens"] - features_frame["num_palavras"],
        features_frame["num_tokens"],
    )
    features_frame["uppercaseRatio"] = calculate_ratio(
        features_frame["num_maiusculas"], features_frame["num_palavras"]
    )
    return features_frame


def extract_anomaly_features(news_frame, character_limit=CHARACTER_LIMIT):
    features_frame = news_frame.copy(deep=True)
    features_frame["num_palavras_original"] = news_frame["num_palavras"]
    features_frame["text_original"] = news_frame["text"]
    features_frame[numeric_metadata_columns] = np.nan
    text_measurements = pd.DataFrame(
        [measure_text(text, character_limit) for text in news_frame["text"]],
        index=news_frame.index,
    )
    for column in text_measurements:
        features_frame[column] = text_measurements[column]
    return add_derived_ratio_features(features_frame)

In [ ]:
features_frame = extract_anomaly_features(news_frame)
features_frame[ANOMALY_COLUMNS] = features_frame[ANOMALY_COLUMNS].replace(
    [np.inf, -np.inf], np.nan
)
missing_counts = features_frame[ANOMALY_COLUMNS].isna().sum().rename("missingCount")
display(missing_counts.to_frame())
print("Limite de caracteres:", CHARACTER_LIMIT)
length_summary = features_frame.groupby("label")[
    ["num_palavras_original", "num_palavras", "num_caracteres"]
].agg(["min", "median", "max"])
display(length_summary)
short_text_count = features_frame["num_caracteres"].lt(CHARACTER_LIMIT).sum()
print("Textos menores que o limite:", int(short_text_count))

## 7. Sanity checks

As seis features utilizam o prefixo; a presença de autor vem dos metadados.
Contagens originais ficam preservadas para auditoria e não alimentam o modelo.

In [ ]:
assert "num_palavras" not in ANOMALY_COLUMNS
assert "label" not in ANOMALY_COLUMNS and "id" not in ANOMALY_COLUMNS
assert len(ANOMALY_COLUMNS) == len(set(ANOMALY_COLUMNS)) == 6
assert features_frame["id"].is_unique
assert features_frame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert features_frame["num_palavras_original"].equals(news_frame["num_palavras"])
assert features_frame["num_verbos"].isna().all()
assert not np.isinf(features_frame[ANOMALY_COLUMNS].to_numpy(dtype=float)).any()
assert measure_text("casa " * 100, 300)["num_palavras"] == 60
assert measure_text("casa " * 100, 300)["num_caracteres"] == 300
print("Contagens do prefixo verificadas; num_palavras não entra no treinamento.")

## 8. Análise estatística

Estatísticas descritivas por label, sem substituir vetores individuais por médias.
Esta inspeção do corpus completo atende ao protocolo exploratório: não usamos
seus resultados para selecionar features, ajustar hiperparâmetros ou threshold.
Qualquer escolha futura guiada por estes resultados exige nova avaliação independente.
Correlações de Pearson com comprimento são mostradas no total e por classe para
evitar confundir efeitos de classe e de tamanho. NaN em correlação pode indicar
feature constante. Mantemos as seis features recalculadas, inclusive possíveis redundâncias.

In [ ]:
statistics_records = []
for label, group in features_frame.groupby("label", sort=True):
    for feature in ANOMALY_COLUMNS:
        values = group[feature]
        first_quartile, third_quartile = values.quantile([0.25, 0.75])
        statistics_records.append({
            "label": label,
            "class": label_names[label],
            "feature": feature,
            "count": values.count(),
            "mean": values.mean(),
            "median": values.median(),
            "std": values.std(),
            "min": values.min(),
            "Q1": first_quartile,
            "Q3": third_quartile,
            "IQR": third_quartile - first_quartile,
            "max": values.max(),
            "missingCount": values.isna().sum(),
        })
feature_statistics = pd.DataFrame(statistics_records).set_index(
    ["label", "class", "feature"]
)
display(feature_statistics)

length_columns = ["num_palavras", "num_tokens"]
correlation_groups = [
    ("All", features_frame),
    ("True (0)", features_frame.loc[features_frame["label"].eq(0)]),
    ("Fake (1)", features_frame.loc[features_frame["label"].eq(1)]),
]
length_correlations = pd.concat({
    name: group[ANOMALY_COLUMNS + length_columns]
    .corr()
    .loc[ANOMALY_COLUMNS, length_columns]
    for name, group in correlation_groups
}, names=["group", "feature"])
display(length_correlations)

feature_correlations = features_frame[ANOMALY_COLUMNS].corr()
display(feature_correlations.round(3))
display(length_correlations.xs("typeTokenRatio", level="feature"))
print(
    "Pearson TTR vs diversidade:",
    feature_correlations.loc["typeTokenRatio", "diversidade"],
)

## 9. Train / Validation / Test

True: 60% treino, 20% validação, 20% teste. Fake: 50% validação, 50% teste.
Todas as divisões usam `random_state=42`. Verificamos IDs exclusivos e cobertura
integral do corpus. **Nenhuma notícia Fake participa de fit ou calibração.**

Limitação do protocolo solicitado: a divisão é por notícia, não por assunto,
fonte ou data. O corpus possui pares True/Fake com o mesmo número-base; IDs
`123t` e `123` são notícias distintas, mas podem tratar do mesmo assunto em
partições diferentes. IDs exclusivos não demonstram independência temática.

In [ ]:
normal_frame = features_frame.loc[features_frame["label"].eq(0)].copy()
fake_frame = features_frame.loc[features_frame["label"].eq(1)].copy()
normal_train_frame, normal_holdout_frame = train_test_split(
    normal_frame, train_size=0.60, random_state=RANDOM_STATE,
)
normal_validation_frame, normal_test_frame = train_test_split(
    normal_holdout_frame, test_size=0.50, random_state=RANDOM_STATE,
)
fake_validation_frame, fake_test_frame = train_test_split(
    fake_frame, test_size=0.50, random_state=RANDOM_STATE,
)
partitions = {
    "normalTrain": normal_train_frame,
    "normalValidation": normal_validation_frame,
    "normalTest": normal_test_frame,
    "fakeValidation": fake_validation_frame,
    "fakeTest": fake_test_frame,
}
for name, frame in partitions.items():
    assert not frame.empty and frame["id"].is_unique
    assert frame["label"].eq(0 if name.startswith("normal") else 1).all()
for (left_name, left), (right_name, right) in combinations(partitions.items(), 2):
    assert set(left["id"]).isdisjoint(right["id"]), f"IDs compartilhados: {left_name}/{right_name}"
assert set().union(*(set(frame["id"]) for frame in partitions.values())) == set(features_frame["id"])
assert sum(len(frame) for frame in partitions.values()) == len(features_frame)
assert normal_train_frame["label"].eq(0).all()
display(pd.DataFrame([
    {"partition": name, "count": len(frame), "label": int(frame["label"].iloc[0])}
    for name, frame in partitions.items()
]).set_index("partition"))

## 10. One-Class SVM e guardas True-only

Cada candidato possui imputer, scaler e detector próprios. O `.fit()` recebe somente
`normal_train_frame`. `nu` é um limite teórico superior para erros de treino e inferior para
vetores de suporte, sujeito ao ajuste e a efeitos numéricos; as frações observadas são medidas.


In [ ]:
ONE_CLASS_CANDIDATES = {
    "ocsvm_nu_001": 0.01,
    "ocsvm_nu_0025": 0.025,
    "ocsvm_nu_005": 0.05,
    "ocsvm_nu_010": 0.10,
}


def build_one_class_pipeline(nu):
    if nu not in ONE_CLASS_CANDIDATES.values():
        raise ValueError(f"nu fora da matriz pré-registrada: {nu}")
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("detector", OneClassSVM(kernel="rbf", gamma="scale", nu=nu)),
    ])


def fit_normal_only(pipeline, training_frame):
    if training_frame.empty or not training_frame["label"].eq(0).all():
        raise ValueError("Treino permitido apenas com notícias True (label == 0).")
    training_features = training_frame[ANOMALY_COLUMNS]
    if np.isinf(training_features.to_numpy(dtype=float)).any():
        raise ValueError("Features de treino contém infinito.")
    empty_columns = training_features.columns[training_features.isna().all()].tolist()
    if empty_columns:
        raise ValueError(f"Features inteiramente ausentes em normal_train: {empty_columns}")
    return pipeline.fit(training_features)


def anomaly_scores(pipeline, frame):
    scores = -pipeline.decision_function(frame[ANOMALY_COLUMNS])
    if not np.isfinite(scores).all():
        raise ValueError("Scores não finitos.")
    return scores


def native_flags(pipeline, frame):
    decision_flags = pipeline.decision_function(frame[ANOMALY_COLUMNS]) < 0
    prediction_flags = pipeline.predict(frame[ANOMALY_COLUMNS]) == -1
    if not np.array_equal(decision_flags, prediction_flags):
        raise AssertionError("Fronteira nativa divergiu de predict == -1.")
    return decision_flags


def binary_metrics(labels, scores, flags):
    labels = np.asarray(labels, dtype=int)
    flags = np.asarray(flags, dtype=bool)
    matrix = confusion_matrix(labels, flags.astype(int), labels=[0, 1])
    tn, fp, fn, tp = (int(value) for value in matrix.ravel())
    total = matrix.sum()
    false_positive_rate = fp / (tn + fp) if tn + fp else 0.0
    return {
        "rocAuc": float(roc_auc_score(labels, scores)),
        "averagePrecision": float(average_precision_score(labels, scores)),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": float((tp + tn) / total),
        "precision": float(precision_score(labels, flags, zero_division=0)),
        "recall": float(recall_score(labels, flags, zero_division=0)),
        "f1": float(f1_score(labels, flags, zero_division=0)),
        "fpr": float(false_positive_rate),
    }


def select_candidate(validation_records, tolerance=1e-12):
    if not validation_records:
        raise ValueError("A seleção exige resultados de validação.")
    best_auc = max(record["validationRocAuc"] for record in validation_records)
    auc_ties = [
        record for record in validation_records
        if best_auc - record["validationRocAuc"] <= tolerance
    ]
    best_ap = max(record["validationAveragePrecision"] for record in auc_ties)
    ap_ties = [
        record for record in auc_ties
        if best_ap - record["validationAveragePrecision"] <= tolerance
    ]
    return min(ap_ties, key=lambda record: record["nu"])["modelKey"]


## 11. Treino, validação e seleção congelada

Os quatro pipelines são ajustados em 2.160 True. O q95 usa exclusivamente as 720 True de
validação. Fake-validation assiste somente a escolha de `nu` por ROC-AUC, AP e menor `nu`.
Nenhuma métrica de teste existe quando `selected_model_key` é materializado.


In [ ]:
validation_frame = pd.concat(
    [normal_validation_frame, fake_validation_frame], ignore_index=True
)
validation_labels = validation_frame["label"].to_numpy(dtype=int)
candidate_pipelines = {}
validation_records = []

for model_key, nu in ONE_CLASS_CANDIDATES.items():
    pipeline = build_one_class_pipeline(nu)
    fit_started = time.perf_counter()
    fit_normal_only(pipeline, normal_train_frame)
    fit_seconds = time.perf_counter() - fit_started

    score_started = time.perf_counter()
    normal_validation_scores = anomaly_scores(pipeline, normal_validation_frame)
    validation_scores = anomaly_scores(pipeline, validation_frame)
    validation_native_flags = native_flags(pipeline, validation_frame)
    score_seconds = time.perf_counter() - score_started

    q95_threshold = float(np.quantile(normal_validation_scores, 0.95))
    validation_q95_flags = validation_scores >= q95_threshold
    native_metrics = binary_metrics(
        validation_labels, validation_scores, validation_native_flags
    )
    q95_metrics = binary_metrics(
        validation_labels, validation_scores, validation_q95_flags
    )
    train_native_flags = native_flags(pipeline, normal_train_frame)
    detector = pipeline["detector"]
    validation_record = {
        "modelKey": model_key,
        "nu": nu,
        "kernel": detector.kernel,
        "resolvedGamma": float(detector._gamma),
        "normalTrainCount": len(normal_train_frame),
        "normalValidationCount": len(normal_validation_frame),
        "fakeValidationCount": len(fake_validation_frame),
        "imputerMedian": pipeline["imputer"].statistics_.astype(float).tolist(),
        "scalerMean": pipeline["scaler"].mean_.astype(float).tolist(),
        "scalerScale": pipeline["scaler"].scale_.astype(float).tolist(),
        "supportVectorCount": int(detector.support_.size),
        "supportVectorFraction": float(detector.support_.size / len(normal_train_frame)),
        "normalTrainNativeOutlierCount": int(train_native_flags.sum()),
        "normalTrainNativeOutlierFraction": float(train_native_flags.mean()),
        "fitSeconds": fit_seconds,
        "validationScoreSeconds": score_seconds,
        "q95Threshold": q95_threshold,
        "normalValidationQ95AlertRate": float(
            np.mean(normal_validation_scores >= q95_threshold)
        ),
        "validationRocAuc": q95_metrics["rocAuc"],
        "validationAveragePrecision": q95_metrics["averagePrecision"],
        "nativeMetrics": native_metrics,
        "q95Metrics": q95_metrics,
    }

    np.testing.assert_allclose(
        pipeline["imputer"].statistics_,
        normal_train_frame[ANOMALY_COLUMNS].median().to_numpy(),
    )
    imputed_train = pipeline["imputer"].transform(
        normal_train_frame[ANOMALY_COLUMNS]
    )
    np.testing.assert_allclose(pipeline["scaler"].mean_, imputed_train.mean(axis=0))
    expected_scale = imputed_train.std(axis=0)
    expected_scale[expected_scale == 0] = 1.0
    np.testing.assert_allclose(pipeline["scaler"].scale_, expected_scale)

    candidate_pipelines[model_key] = pipeline
    validation_records.append(validation_record)

selected_model_key = select_candidate(validation_records)
selected_pipeline = candidate_pipelines[selected_model_key]
selected_validation_record = next(
    record for record in validation_records
    if record["modelKey"] == selected_model_key
)
selected_q95_threshold = selected_validation_record["q95Threshold"]
validation_results_frame = pd.DataFrame(validation_records)
result_columns = [
    "modelKey",
    "nu",
    "resolvedGamma",
    "supportVectorCount",
    "supportVectorFraction",
    "normalTrainNativeOutlierFraction",
    "q95Threshold",
    "normalValidationQ95AlertRate",
    "validationRocAuc",
    "validationAveragePrecision",
]
display(validation_results_frame[result_columns])
print("Candidato congelado antes do teste:", selected_model_key)


## 12. Controle Isolation Forest e avaliação final única

O controle usa o mesmo treino, features, IDs e q95 True-validation. Somente o candidato OCSVM
selecionado recebe métricas de teste. A fronteira nativa do OCSVM permanece separada do q95.


In [ ]:
control_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    (
        "detector",
        IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])
fit_normal_only(control_pipeline, normal_train_frame)
control_normal_validation_scores = anomaly_scores(
    control_pipeline, normal_validation_frame
)
control_q95_threshold = float(
    np.quantile(control_normal_validation_scores, 0.95)
)

test_frame = pd.concat([normal_test_frame, fake_test_frame], ignore_index=True)
test_labels = test_frame["label"].to_numpy(dtype=int)
selected_scores = anomaly_scores(selected_pipeline, test_frame)
selected_native_flags = native_flags(selected_pipeline, test_frame)
selected_q95_flags = selected_scores >= selected_q95_threshold
control_scores = anomaly_scores(control_pipeline, test_frame)
control_q95_flags = control_scores >= control_q95_threshold

selected_native_metrics = binary_metrics(
    test_labels, selected_scores, selected_native_flags
)
selected_q95_metrics = binary_metrics(test_labels, selected_scores, selected_q95_flags)
control_q95_metrics = binary_metrics(test_labels, control_scores, control_q95_flags)
score_frames = {
    "ocsvm": pd.DataFrame({"label": test_labels, "score": selected_scores}),
    "isolationForest": pd.DataFrame({"label": test_labels, "score": control_scores}),
}
score_distributions = pd.concat({
    model: scores.groupby("label")["score"].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
    for model, scores in score_frames.items()
}, names=["model", "label"])

test_predictions_frame = test_frame[["id", "label"] + ANOMALY_COLUMNS].copy()
test_predictions_frame["ocsvmAnomalyScore"] = selected_scores
test_predictions_frame["ocsvmNativeIsAnomaly"] = selected_native_flags
test_predictions_frame["ocsvmQ95IsAnomaly"] = selected_q95_flags
test_predictions_frame["isolationForestAnomalyScore"] = control_scores
test_predictions_frame["isolationForestQ95IsAnomaly"] = control_q95_flags
assert test_predictions_frame["id"].is_unique

transition_conditions = [
    selected_q95_flags & control_q95_flags,
    ~selected_q95_flags & ~control_q95_flags,
    selected_q95_flags & ~control_q95_flags,
]
transition_labels = np.select(
    transition_conditions,
    ["alerta_mantido", "sem_alerta", "alerta_adicionado_ocsvm"],
    default="alerta_removido_ocsvm",
)
test_predictions_frame["transition"] = transition_labels
transition_counts = (
    test_predictions_frame.groupby(["label", "transition"])
    .size()
    .rename("count")
    .reset_index()
)
assert int(transition_counts["count"].sum()) == len(test_predictions_frame)

test_results = {
    "selectedModelKey": selected_model_key,
    "fakePrevalenceApBaseline": float(np.mean(test_labels == 1)),
    "ocsvmQ95Threshold": selected_q95_threshold,
    "isolationForestQ95Threshold": control_q95_threshold,
    "ocsvmNative": selected_native_metrics,
    "ocsvmQ95": selected_q95_metrics,
    "isolationForestQ95": control_q95_metrics,
    "transitions": transition_counts.to_dict(orient="records"),
}
display(score_distributions)
metric_comparison = pd.DataFrame({
    "OCSVM nativo": selected_native_metrics,
    "OCSVM q95": selected_q95_metrics,
    "Isolation Forest q95": control_q95_metrics,
}).T
display(metric_comparison)
display(transition_counts)


## 13. Exportação opcional e conclusão calculada

Quando `ONE_CLASS_SVM_OUTPUT_DIR` existe, o `Run All` salva resultados estruturados nessa
pasta nova. Sem a variável, o notebook permanece interativo e não grava artefatos experimentais.


In [ ]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(element) for key, element in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(element) for element in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    return value


manifest = {
    "experimentKey": EXPERIMENT_KEY,
    "baseNotebookSha256": BASE_NOTEBOOK_SHA256,
    "corpusRevision": CORPUS_REVISION,
    "seed": RANDOM_STATE,
    "characterLimit": CHARACTER_LIMIT,
    "features": ANOMALY_COLUMNS,
    "partitionCounts": {name: len(frame) for name, frame in partitions.items()},
    "candidates": ONE_CLASS_CANDIDATES,
    "selectedModelKey": selected_model_key,
    "selectionUsesFakeValidationLabels": True,
    "testWasNotUsedForSelection": True,
}
output_directory = os.environ.get("ONE_CLASS_SVM_OUTPUT_DIR")
if output_directory:
    output_path = Path(output_directory)
    output_path.mkdir(parents=True, exist_ok=False) if not output_path.exists() else None
    (output_path / "manifest.json").write_text(json.dumps(
        json_ready(manifest), indent=2, allow_nan=False
    ), encoding="utf-8")
    (output_path / "validation-results.json").write_text(json.dumps(
        json_ready(validation_records), indent=2, allow_nan=False
    ), encoding="utf-8")
    (output_path / "test-results.json").write_text(json.dumps(
        json_ready(test_results), indent=2, allow_nan=False
    ), encoding="utf-8")
    test_predictions_frame.to_csv(output_path / "test-predictions.csv", index=False)
    transition_counts.to_csv(output_path / "transitions.csv", index=False)

display(Markdown(f'''O candidato congelado foi **{selected_model_key}**. No teste histórico, OCSVM q95 obteve
ROC-AUC **{selected_q95_metrics['rocAuc']:.4f}**, AP **{selected_q95_metrics['averagePrecision']:.4f}**,
recall **{selected_q95_metrics['recall']:.2%}** e FPR **{selected_q95_metrics['fpr']:.2%}**.
O controle Isolation Forest q95 obteve ROC-AUC **{control_q95_metrics['rocAuc']:.4f}**,
AP **{control_q95_metrics['averagePrecision']:.4f}**, recall **{control_q95_metrics['recall']:.2%}**
e FPR **{control_q95_metrics['fpr']:.2%}**.

Estes números descrevem desvio linguístico neste split histórico; não provam falsidade nem
superioridade estatística. A seleção de `nu` foi assistida por labels Fake de validação. O texto
foi truncado em 300 caracteres, o split não é temático e o teste histórico já foi consultado.
**Colab não foi validado neste recorte.**
'''))
